In [1]:
!pip install -q -U google-genai pypdf
print("Libraries installed successfully")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 16.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.61.0 which is incompatible.
Libraries installed successfully


In [2]:
import numpy as np

from google import genai
from google.genai import types
from google.colab import userdata


client = genai.Client(
    api_key=userdata.get("GEMINI_API")
)

print('setup done')

setup done


In [3]:
from google.colab import files

uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]
print("Uploaded:", pdf_name)

Saving POLICY ON DSB SERVICES 5.0.pdf to POLICY ON DSB SERVICES 5.0.pdf
Uploaded: POLICY ON DSB SERVICES 5.0.pdf


In [4]:
from pypdf import PdfReader

reader = PdfReader(pdf_name)

text = ""
for page in reader.pages:
    text += (page.extract_text() or "") + "\n"

print("Number of pages:", len(reader.pages))
print("Total characters:", len(text))
print("\nSample text:\n")
print(text[:500])

assert text.strip(), "No text was extracted from the PDF."

Number of pages: 14
Total characters: 25967

Sample text:

1 | P a g e  
 
  
 
 
 
 
POLICY ON DOORSTEP BANKING SERVICES FOR INDIVIDUAL CUSTOMERS 
 
(Version 5.0) 
 
 
 
20.09.2024 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Resources Business Unit 
Corporate Centre 
Mumbai 
(SBI Times>Manuals/Master Circular/SOP/Policies>Policy) 
 
 
 
  
  
  
2 | P a g e  
 
 
 
 
 
Index:  
  
Section   Description  Page No  
  Preamble  3 
  RBI Guidelines  3 
A  Definitions  4 
B  Eligibility  4 
C  Outsourcing of Services  5 
D  Request for Service and Mode of Deli


In [5]:
def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])

        if end >= len(text):
            break

        start = end - overlap

    return chunks


chunks = chunk_text(text, chunk_size=800, overlap=150)

print("Chunk size: 800 characters")
print("Overlap: 150 characters")
print("Number of chunks:", len(chunks))
print("\nOne complete chunk:\n")
print(chunks[0])

Chunk size: 800 characters
Overlap: 150 characters
Number of chunks: 40

One complete chunk:

1 | P a g e  
 
  
 
 
 
 
POLICY ON DOORSTEP BANKING SERVICES FOR INDIVIDUAL CUSTOMERS 
 
(Version 5.0) 
 
 
 
20.09.2024 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Resources Business Unit 
Corporate Centre 
Mumbai 
(SBI Times>Manuals/Master Circular/SOP/Policies>Policy) 
 
 
 
  
  
  
2 | P a g e  
 
 
 
 
 
Index:  
  
Section   Description  Page No  
  Preamble  3 
  RBI Guidelines  3 
A  Definitions  4 
B  Eligibility  4 
C  Outsourcing of Services  5 
D  Request for Service and Mode of Delivery   6 
E  Service Charges  8 
F  Notice of Revision of Charges  8 
G  Redressal of Grievance  9 
H  Risk Mitigation  9 
I  Review of the Policy  10 
J  Ownership of the Policy  10 
ANNEXURE    
1  Regulatory Guidelines   11-14 
    
  
  
  
  
  
   
  
  
  
  
  
3 | P a g e  
 
  
PREAMBLE  



In [6]:
EMB_MODEL = "models/gemini-embedding-001"

def embed_text(t):
    result = client.models.embed_content(
        model=EMB_MODEL,
        contents=t,
        config=types.EmbedContentConfig(
            output_dimensionality=768
        )
    )
    return np.array(result.embeddings[0].values)


chunk_embeddings = []
chunk_store = []

for i, ch in enumerate(chunks):
    embedding = embed_text(ch)

    chunk_embeddings.append(embedding)

    chunk_store.append({
        "chunk_id": i + 1,
        "text": ch,
        "embedding": embedding
    })

    print(
        f"Embedded chunk {i + 1}/{len(chunks)}",
        end="\r"
    )

chunk_embeddings = np.array(chunk_embeddings)

print("\nShape:", chunk_embeddings.shape)
print("Stored text + embedding pairs:", len(chunk_store))

Embedded chunk 40/40
Shape: (40, 768)
Stored text + embedding pairs: 40


In [7]:
chunk_embeddings = chunk_embeddings / np.linalg.norm(
    chunk_embeddings, axis=1, keepdims=True
)

def retrieve(question, k=3):
    q_emb = embed_text(question)
    q_emb = q_emb / np.linalg.norm(q_emb)

    scores = chunk_embeddings @ q_emb

    top_idx = np.argsort(scores)[::-1][:k]

    return [
        (int(i) + 1, chunks[i], scores[i])
        for i in top_idx
    ]


question = input("Ask a question about the SBI policy: ")

results = retrieve(question)

for chunk_id, chunk, score in results:
    print(f"Chunk ID: {chunk_id}")
    print(f"Score: {score:.3f}")
    print(chunk)
    print("-" * 60)

Ask a question about the SBI policy: What is the completion timeline for mandatory doorstep banking services?
Chunk ID: 40
Score: 0.748
y and those 
where it will be provided on a best effort basis and make the policy public. The list 
of branches offering such doorstep banking services shall be displayed/updated 
on the bank’s website regularly.  
    
ii. Banks shall give adequate publicity to the availability of these services in their 
public awareness campaigns. The charges, in this regard, shall also be 
prominently indicated in brochures and published in their websites.  
  
iii. Banks shall report the progress made in this regard to the Customer Service 
Committee of the Board every quarter.  
  
           _________________________  

------------------------------------------------------------
Chunk ID: 3
Score: 0.739
en under the 
aegis of Dept of Financial Services, Govt of India/ Indian Banks’ Association or any 
empowered body, at the industry level (in whole or in part).

I read the three retrieved chunks and confirmed that Chunk 15 contains the correct answer: mandatory doorstep banking services must be completed within T+1 working day, excluding holidays. Its similarity score was 0.735.



In [8]:
MODEL = "gemini-2.5-flash"

ANSWER_CONFIG = types.GenerateContentConfig(
    system_instruction=(
        "Answer using only the provided context. "
        "Never guess or use outside knowledge. "
        "If the answer is not in the context, reply exactly: "
        "'I could not find this in the document.' "
        "If the answer is available, cite the supporting "
        "chunk IDs using the format [Chunk 15]."
    ),
    temperature=0,
    max_output_tokens=500
)


def ask(question, k=3):
    top = retrieve(question, k)


    context = "\n\n-----------\n\n".join(
        f"[Chunk {chunk_id}]\n{chunk}"
        for chunk_id, chunk, score in top
    )

    prompt = f"""CONTEXT FROM THE DOCUMENT:

{context}

QUESTION: {question}

Answer using only the context above.
Cite the supporting chunk IDs.
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=ANSWER_CONFIG
    )

    answer = response.text or ""
    print("\nAnswer:", answer)

    print("\nSupporting chunks and scores:")

    found = False
    for chunk_id, chunk, score in top:
        if f"[Chunk {chunk_id}]" in answer:
            print(f"Chunk {chunk_id} | Score: {score:.3f}")
            found = True

    if not found:
        print("No supporting chunk cited.")

    return answer


print("SBI Policy Bot — type 'exit' to stop.")

while True:
    question = input("\nYour question: ").strip()

    if question.lower() == "exit":
        print("Chat ended.")
        break

    if not question:
        print("Please enter a question.")
        continue

    answer = ask(question)
    print("-" * 60)

SBI Policy Bot — type 'exit' to stop.

Your question: What is the completion timeline for mandatory doorstep banking services?

Answer: Doorstep Banking Services, wherever available mandatory, shall be completed expeditiously but not later than T+1 working day (holidays excluded). [Chunk 15]

Supporting chunks and scores:
Chunk 15 | Score: 0.735
------------------------------------------------------------

Your question: Are minors’ accounts eligible for doorstep banking services?

Answer: No, minors’ accounts, including those under guardianship, are not eligible for doorstep banking services. [Chunk 7]

Supporting chunks and scores:
Chunk 7 | Score: 0.778
------------------------------------------------------------

Your question: How often should the Doorstep Banking Policy Version 5.0 be reviewed?

Answer: The Policy on Doorstep Banking for Individual Customers (Version 5.0) shall be reviewed as and when felt necessary or at least at a two-year interval by the Resources Business Uni

The bot correctly answered all three questions whose answers were available in the PDF, citing Chunks 15, 7 and 25. It correctly refused the three questions whose answers were absent. The cited answers match the policy’s timeline, eligibility and review rules.

In [9]:
chunks_800 = chunks
embeddings_800 = chunk_embeddings


chunks = chunk_text(text, chunk_size=3000, overlap=150)

print("800-character chunk count:", len(chunks_800))
print("3000-character chunk count:", len(chunks))

chunk_embeddings = []

for i, ch in enumerate(chunks):
    chunk_embeddings.append(embed_text(ch))
    print(
        f"Embedded chunk {i + 1}/{len(chunks)}",
        end="\r"
    )

chunk_embeddings = np.array(chunk_embeddings)

chunk_embeddings = chunk_embeddings / np.maximum(
    np.linalg.norm(
        chunk_embeddings, axis=1, keepdims=True
    ),
    1e-12
)

print("\nEmbedding shape:", chunk_embeddings.shape)

800-character chunk count: 40
3000-character chunk count: 10
Embedded chunk 10/10
Embedding shape: (10, 768)


In [10]:
print("Experiment — 3000-character chunks")
print("Ask the same three in-document questions.")
print("Type 'exit' when finished.")

while True:
    question = input("\nYour question: ").strip()

    if question.lower() == "exit":
        print("Experiment ended.")
        break

    if not question:
        print("Please enter a question.")
        continue

    answer = ask(question)
    print("-" * 60)

Experiment — 3000-character chunks
Ask the same three in-document questions.
Type 'exit' when finished.

Your question: What is the completion timeline for mandatory doorstep banking services?

Answer: I could not find this in the document.

Supporting chunks and scores:
No supporting chunk cited.
------------------------------------------------------------

Your question: Are minors’ accounts eligible for doorstep banking services?

Answer: No, minors’ accounts, including those under guardianship, are not eligible for doorstep banking services. [Chunk 2]

Supporting chunks and scores:
Chunk 2 | Score: 0.734
------------------------------------------------------------

Your question: How often should the Doorstep Banking Policy Version 5.0 be reviewed?

Answer: The Doorstep Banking Policy Version 5.0 should be reviewed as and when felt necessary or at least at a two-year interval. [Chunk 6]

Supporting chunks and scores:
Chunk 6 | Score: 0.751
------------------------------------------

Increasing the chunk size from 800 to 3000 reduced the number of chunks from 40 to 10.
The eligibility answer’s similarity score decreased from 0.778 to 0.734, and the review answer’s score decreased from 0.782 to 0.751.
These two answers remained correct, but the bot failed to answer the timeline question that it previously answered correctly.
Larger chunks contain more topics, which may make their embeddings less focused and affect retrieval.
Overall, 800-character chunks worked better for these questions; the retrieved 3000-character chunks need inspection to confirm why the timeline answer was missed.